# 03b — Amazon Interaction Loop

**input:** `data/amazon/items.pkl`, `data/amazon/embeddings.npy`, `data/amazon/Clothing_Shoes_and_Jewelry.jsonl.gz`

Same pipeline as notebook 03 — applied to Amazon Clothing.

**output:** `data/amazon/results_ours.pkl`

In [ ]:
import numpy as np
import pandas as pd
import pickle, gzip, json, sys
from pathlib import Path
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

sys.path.append(str(Path('../')))
from query_templates_amazon import get_query_for_item

DATA_DIR    = Path('../data/amazon')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

items      = pd.read_pickle(DATA_DIR / 'items.pkl')
embeddings = np.load(DATA_DIR / 'embeddings.npy')
sbert      = SentenceTransformer('all-MiniLM-L6-v2')

TOP_N     = 500
STOP_SIZE = 5
MAX_TURNS = 6
N_EVAL    = 200

np.random.seed(42)
print(f'Items: {len(items)}')
print(f'Embeddings: {embeddings.shape}')
print('imports OK')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Items: 9999
Embeddings: (9999, 384)
imports OK


### Part 1 — Load ratings as ground truth

In [ ]:
REVIEW_PATH = DATA_DIR / 'Clothing_Shoes_and_Jewelry.jsonl.gz'

# asin → positional index in items dataframe
asin_to_idx = {asin: idx for idx, asin in enumerate(items['asin'])}

# load only high-rated items (rating >= 4)
high_ratings = []
print('Reading reviews...')
with gzip.open(REVIEW_PATH, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            r = json.loads(line)
            rating = float(r.get('rating', 0))
            asin   = r.get('parent_asin', r.get('asin', ''))
            if rating >= 4 and asin in asin_to_idx:
                high_ratings.append({
                    'asin':      asin,
                    'item_idx':  asin_to_idx[asin],
                    'rating':    rating,
                })
        except:
            continue
        if (i+1) % 500_000 == 0:
            print(f'  Processed {i+1:,} reviews, found {len(high_ratings)} high-rated...')
        # stop once we have enough targets
        if len(high_ratings) >= 5000:
            break

df_ratings = pd.DataFrame(high_ratings).drop_duplicates(subset='item_idx')
print(f'\nHigh-rated items found: {len(df_ratings)}')

# sample N_EVAL targets
sample     = df_ratings.sample(n=min(N_EVAL, len(df_ratings)), random_state=42)
target_items = sample['item_idx'].values

print(f'Targets sampled: {len(target_items)}')
print('\nExample targets:')
for idx in target_items[:3]:
    print(f'  {items.iloc[idx]["title"][:60]}  |  {items.iloc[idx]["category"]}')

Reading reviews...

High-rated items found: 1546
Targets sampled: 200

Example targets:
  Chef Designs Men's Baggy Chef Pant  |  Clothing > Food Service > Chef Pants
  Purses and Handbags Women Tote Shoulder Top Handle Satchel H  |  Women > Handbags & Wallets > Satchels
  IZOD Men's Button Down Long Sleeve Stretch Performance Tatte  |  Clothing > Shirts > Casual Button-Down Shirts


### Part 2 — helper functions

In [ ]:
import sys
sys.path.insert(0, '../src')
from core import (
    entropy,
    information_gain,
    adaptive_k,
    cluster_candidates,
    simulated_user,
    retrieve_candidates,
)

def make_query_embedding(target_idx):
    category  = items.iloc[target_idx]['category']
    query_text = get_query_for_item(category)
    query_emb  = sbert.encode(query_text, normalize_embeddings=True)
    return query_emb, query_text



print('functions OK')

functions OK


### Part 3 — Interaction loop + test

In [ ]:
def run_interaction(target_idx, verbose=False):
    q_emb, q_text = make_query_embedding(target_idx)
    C = retrieve_candidates(q_emb, TOP_N)
    if target_idx not in C:
        C.append(target_idx)

    entropy_trace = [entropy(C)]
    ig_trace = []
    turn = 0

    while len(C) > STOP_SIZE and turn < MAX_TURNS:
        k = adaptive_k(len(C))
        partition = cluster_candidates(C, k)
        ig = information_gain(C, partition)
        ig_trace.append(ig)
        if verbose:
            print(f'  Turn {turn+1}: |C|={len(C)}, k={len(partition)}, IG={ig:.3f}')
        C = partition[simulated_user(partition, target_idx)]
        turn += 1
        entropy_trace.append(entropy(C))

    return {
        'turns':         turn,
        'final_size':    len(C),
        'success':       target_idx in C,
        'entropy_trace': entropy_trace,
        'ig_trace':      ig_trace,
        'query_text':    q_text,
    }


# ── test on a single item ──────────────────────────────────
target = int(target_items[0])
r = run_interaction(target, verbose=True)
print()
print(f'Item   : {items.iloc[target]["title"][:60]}')
print(f'Category: {items.iloc[target]["category"]}')
print(f'Query  : {r["query_text"]}')
print(f'Turns  : {r["turns"]}')
print(f'Final  : {r["final_size"]} items')
print(f'Success: {r["success"]}')

  Turn 1: |C|=501, k=5, IG=2.254
  Turn 2: |C|=159, k=5, IG=1.948
  Turn 3: |C|=60, k=4, IG=1.862
  Turn 4: |C|=23, k=3, IG=1.121

Item   : Chef Designs Men's Baggy Chef Pant
Category: Clothing > Food Service > Chef Pants
Query  : I need shoes that go with everything
Turns  : 4
Final  : 1 items
Success: True


### Part 4 — Evaluation

In [ ]:
results = []
for t in tqdm(target_items, desc='Amazon — Our method'):
    r = run_interaction(int(t))
    r['target_idx'] = int(t)
    results.append(r)

df = pd.DataFrame(results)

print('\n── Amazon Results ───────────────────────────')
print(f'Avg turns     : {df["turns"].mean():.2f}')
print(f'Avg final size: {df["final_size"].mean():.2f}')
print(f'Success rate  : {df["success"].mean():.3f}')

with open(DATA_DIR / 'results_ours.pkl', 'wb') as f:
    pickle.dump({'df': df, 'target_items': target_items}, f)

print('Saved: data/amazon/results_ours.pkl')
print()
print('✅ notebook 03b complete')

Amazon — Our method: 100%|██████████| 200/200 [00:11<00:00, 17.67it/s]


── Amazon Results ───────────────────────────
Avg turns     : 4.26
Avg final size: 3.35
Success rate  : 1.000
Saved: data/amazon/results_ours.pkl

✅ notebook 03b complete
